In [1]:
import os
from pathlib import Path
from tqdm import tqdm

from beak.experimental.utilities.raster_processing import unify_raster_grids
from beak.experimental.utilities.io import save_raster, create_file_list, check_path, data_folder


**User inputs**
1. Select the root folder where the rasters are stored.
2. Then, go down to select the subfolders that contain the rasters to be unified.<p>

The script will search for all rasters and store them in relative paths.

In [2]:
BASE_PATH = data_folder()
EPSG_CODE = 102008
RESOLUTION = 500

BASE_EXTENT = "tusk_great_basin"
BASE_RASTER = BASE_PATH / "processed" / str(f"regional_{BASE_EXTENT}_{EPSG_CODE}_{RESOLUTION}") / "base_raster" / "template_raster.tif"
BASE_EVIDENCE = "remote_sensing"
CMA_EXTENT = "regional"

# Export
# If some data sets are already **LOG** scaled they need special actions. They will be unified and stored in a separate folder.
PATH_INPUT = BASE_PATH / "raw" / BASE_EVIDENCE / "regional_scale" / "laculi_southwest"
PATH_EXPORT = BASE_PATH / "processed" / str(f"{CMA_EXTENT}_{BASE_EXTENT}_{EPSG_CODE}_{RESOLUTION}") / "unified" / BASE_EVIDENCE
CURRENT_DIR = Path(os.getcwd()) / PATH_EXPORT.name
OUT_FOLDER = PATH_EXPORT

print(f"Input folder: {PATH_INPUT}")
print(f"Export folder: {OUT_FOLDER}")

Input folder: S:\Projekte\20230082_DARPA_CriticalMAAS_TA3\Bearbeitung\GitHub\beak-ta3\src\beak\data\raw\remote_sensing\regional_scale\laculi_southwest
Export folder: S:\Projekte\20230082_DARPA_CriticalMAAS_TA3\Bearbeitung\GitHub\beak-ta3\src\beak\data\processed\regional_tusk_great_basin_102008_500\unified\remote_sensing


Select subfolders to be scaled.

In [3]:
for folder in os.listdir(PATH_INPUT):
  if os.path.isdir(os.path.join(PATH_INPUT, folder)):
    print(folder)


spectral


In [4]:
SELECTION = [
    "spectral",
    ]

input_folders = [PATH_INPUT / folder for folder in SELECTION]

print("Selected folders:")
for folder in input_folders:
  print(folder)

Selected folders:
S:\Projekte\20230082_DARPA_CriticalMAAS_TA3\Bearbeitung\GitHub\beak-ta3\src\beak\data\raw\remote_sensing\regional_scale\laculi_southwest\spectral


**Select files**

In [5]:
# Create file list
file_list = []
for folder in input_folders:
  files = create_file_list(folder, recursive=True)
  file_list.extend(files)

# Show results
print(f"Found {len(file_list)} files.")


Found 12 files.


**Run unification**

In [13]:
for file in tqdm(file_list, total=len(file_list)):
    folder_relative = os.path.relpath(file.parent, PATH_INPUT)
        
    unified_raster, unified_meta = unify_raster_grids(BASE_RASTER, [file], resampling_mode="manual", same_extent=True, same_shape=True)[0]
    
    unified_meta.update({
        "dtype": "int8",
        "nodata": -99
        }
    )
    
    output_folder = OUT_FOLDER / str(folder_relative)
    out_path = output_folder / file.name
    check_path(Path(os.path.dirname(out_path)))
    
    # Raises a warning since the CDR template raster is float32 and has a int-incompatible nodata value
    save_raster(out_path, array=unified_raster, dtype="int8", metadata=unified_meta, overwrite=True, nodata_value=-99)


  0%|          | 0/12 [00:00<?, ?it/s]S:\Projekte\20230082_DARPA_CriticalMAAS_TA3\Bearbeitung\GitHub\beak-ta3\src\beak\utilities\io.py:370: RuntimeWarning: invalid value encountered in cast
  dst.write(array[i].astype(dtype), i + 1)
  8%|▊         | 1/12 [00:03<00:43,  3.91s/it]S:\Projekte\20230082_DARPA_CriticalMAAS_TA3\Bearbeitung\GitHub\beak-ta3\src\beak\utilities\io.py:370: RuntimeWarning: invalid value encountered in cast
  dst.write(array[i].astype(dtype), i + 1)
 17%|█▋        | 2/12 [00:08<00:41,  4.13s/it]S:\Projekte\20230082_DARPA_CriticalMAAS_TA3\Bearbeitung\GitHub\beak-ta3\src\beak\utilities\io.py:370: RuntimeWarning: invalid value encountered in cast
  dst.write(array[i].astype(dtype), i + 1)
 25%|██▌       | 3/12 [00:12<00:37,  4.21s/it]S:\Projekte\20230082_DARPA_CriticalMAAS_TA3\Bearbeitung\GitHub\beak-ta3\src\beak\utilities\io.py:370: RuntimeWarning: invalid value encountered in cast
  dst.write(array[i].astype(dtype), i + 1)
 33%|███▎      | 4/12 [00:16<00:33,  4.17s/i